# 🛡️ TP : Introduction aux API LLM pour l'analyse de logs cybersécurité

**Objectif** : Apprendre à utiliser l'API d'Anthropic (Claude) pour analyser automatiquement des logs de sécurité.

---


## 1. Installation et configuration

In [ ]:
# !pip install anthropic

In [ ]:
# Installation de la bibliothèque Anthropic
# Décommentez si nécessaire
# !pip install anthropic

import anthropic # importe le SDK Anthropic pour interagir avec les modèles de langage
import os
from dotenv import load_dotenv
load_dotenv()  # Charge les variables d'environnement depuis un fichier .env (optionnel)

# ✅ La clé API est lue depuis une variable d'environnement (jamais en dur dans le code)
# Avant de lancer ce notebook, exécutez dans votre terminal :
#   export ANTHROPIC_API_KEY="sk-ant-..."
API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "⚠️  Variable ANTHROPIC_API_KEY introuvable.\n"
        "Lancez : export ANTHROPIC_API_KEY='votre_clé' dans votre terminal."
    )




✓ Configuration terminée !


In [ ]:
# Initialisation du client (sans passer la clé explicitement : le SDK la lit seul)
client = anthropic.Anthropic(api_key=API_KEY )          # Lit ANTHROPIC_API_KEY automatiquement

print("✓ Configuration terminée !")

✓ Configuration terminée !


## 2. Fonction d'appel API

In [14]:
?client.messages.create

Signature:
client.messages.create(
    *,
    max_tokens: 'int',
    messages: 'Iterable[MessageParam]',
    model: 'ModelParam',
    cache_control: 'Optional[CacheControlEphemeralParam] | Omit' = <anthropic.Omit object at 0x00000235D22C0970>,
    container: 'Optional[str] | Omit' = <anthropic.Omit object at 0x00000235D22C0970>,
    inference_geo: 'Optional[str] | Omit' = <anthropic.Omit object at 0x00000235D22C0970>,
    metadata: 'MetadataParam | Omit' = <anthropic.Omit object at 0x00000235D22C0970>,
    output_config: 'OutputConfigParam | Omit' = <anthropic.Omit object at 0x00000235D22C0970>,
    service_tier: "Literal['auto', 'standard_only'] | Omit" = <anthropic.Omit object at 0x00000235D22C0970>,
    stop_sequences: 'SequenceNotStr[str] | Omit' = <anthropic.Omit object at 0x00000235D22C0970>,
    stream: 'Literal[False] | Literal[True] | Omit' = <anthropic.Omit object at 0x00000235D22C0970>,
    system: 'Union[str, Iterable[TextBlockParam]] | Omit' = <anthropic.Omit object at 0x0

In [17]:
def appeler_claude(prompt: str, max_tokens: int = 1000, temperature: float = 0.0) -> str:
    """
    Appelle l'API Claude et retourne le texte de la réponse.
    Lève une exception en cas d'erreur (ne swallow pas silencieusement).
    """
    response = client.messages.create(
        model="claude-opus-4-6",          # ✅ Modèle récent
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.content[0].text

# Test rapide



In [29]:
prompt = "Dis 'bonjour, où se trouve la poste centrale?' en ewondo"

In [30]:
print("Test de connexion :")
print(appeler_claude(prompt=prompt))

Test de connexion :
En ewondo, on peut dire :

**« Mbolo, post nnam a bé hé ? »**

Décomposition :
- **Mbolo** = Bonjour (salutation courante)
- **Post nnam** = poste centrale (bureau de poste principal)
- **a bé hé ?** = elle se trouve où ?

> **Note :** L'ewondo est une langue bantoue parlée principalement au Cameroun (région du Centre). Certains termes modernes comme « poste » sont souvent empruntés au français et intégrés dans la phrase ewondo. Les variantes peuvent exister selon les locuteurs et les contextes régionaux.


In [31]:
print(appeler_claude(prompt=prompt, temperature=0.7))

En ewondo, on peut dire :

**« Mbolo, post nnam a bé hé ? »**

ou plus couramment :

**« Mbolo, posté ngbwa a bé vé ? »**

Cependant, il faut noter que l'ewondo, comme beaucoup de langues bantoues du Cameroun, a intégré certains emprunts au français pour des concepts modernes comme « la poste ». La traduction la plus naturelle serait :

**« Mbolo, posté e nnam e bé hé ? »**

- **Mbolo** = Bonjour
- **Posté e nnam** = la poste centrale (littéralement « la poste principale/du pays »)
- **e bé hé ?** = elle se trouve où ?

> ⚠️ L'ewondo étant une langue essentiellement orale avec des variations selon les locuteurs et les régions, je vous recommande de vérifier auprès d'un locuteur natif pour une formulation plus précise et naturelle.


In [32]:
print(appeler_claude(prompt=prompt, temperature=0.1))

En ewondo, on peut dire :

**« Mbolo, post nnam a bé hé ? »**

Ou plus couramment :

**« Mbolo, poste e nnam e bé he ? »**

Voici le détail :
- **Mbolo** = Bonjour (salutation courante en ewondo)
- **Poste e nnam** = la poste centrale (littéralement « la poste du pays/principale »)
- **e bé he ?** = elle se trouve où ?

> **Note** : L'ewondo étant une langue principalement orale avec des variations selon les locuteurs et les régions du Centre-Cameroun, il peut exister des formulations légèrement différentes selon les interlocuteurs. Le mot « poste » est souvent emprunté directement au français, comme c'est courant pour les termes administratifs modernes.


## 3. Exemple 1 – Analyse d'un log SSH suspect

In [33]:
log_ssh = """
2024-01-15 03:23:45 auth.log: Failed password for root from 192.168.1.100 port 54322 ssh2
2024-01-15 03:23:47 auth.log: Failed password for root from 192.168.1.100 port 54324 ssh2
2024-01-15 03:23:49 auth.log: Failed password for root from 192.168.1.100 port 54326 ssh2
2024-01-15 03:23:51 auth.log: Failed password for root from 192.168.1.100 port 54328 ssh2
2024-01-15 03:23:53 auth.log: Failed password for root from 192.168.1.100 port 54330 ssh2
"""

In [34]:
prompt_analyse = f"""
Tu es un expert en cybersécurité. Analyse ce log SSH et réponds aux questions :
1. Quel est le type d'attaque potentielle ?
2. Quelle est l'adresse IP source ?
3. Quelles actions recommandes-tu ?

Log :
{log_ssh}
"""

In [35]:
print(prompt_analyse)


Tu es un expert en cybersécurité. Analyse ce log SSH et réponds aux questions :
1. Quel est le type d'attaque potentielle ?
2. Quelle est l'adresse IP source ?
3. Quelles actions recommandes-tu ?

Log :

2024-01-15 03:23:45 auth.log: Failed password for root from 192.168.1.100 port 54322 ssh2
2024-01-15 03:23:47 auth.log: Failed password for root from 192.168.1.100 port 54324 ssh2
2024-01-15 03:23:49 auth.log: Failed password for root from 192.168.1.100 port 54326 ssh2
2024-01-15 03:23:51 auth.log: Failed password for root from 192.168.1.100 port 54328 ssh2
2024-01-15 03:23:53 auth.log: Failed password for root from 192.168.1.100 port 54330 ssh2




In [37]:


print(appeler_claude(prompt= prompt_analyse, temperature=0.7))


# 🔍 Analyse du log SSH

## 1. Type d'attaque potentielle

### **Attaque par force brute (Brute Force Attack) sur SSH**

Plusieurs indicateurs convergent :

| Indicateur | Détail | Niveau de suspicion |
|---|---|---|
| **Répétition rapide** | 5 tentatives en **8 secondes** (~1 toutes les 2s) | 🔴 Élevé |
| **Compte ciblé** | `root` — le compte le plus privilégié | 🔴 Critique |
| **Échecs systématiques** | 100% de `Failed password` | 🔴 Élevé |
| **Ports source incrémentaux** | 54322 → 54324 → 54326 → 54328 → 54330 (+2) | 🔴 Automatisation |
| **Horaire suspect** | **03h23 du matin** — heure creuse typique | 🟡 Moyen |

> L'incrémentation régulière des ports source (+2) est la signature caractéristique d'un **outil automatisé** (type Hydra, Medusa, Ncrack) et non d'un humain.

---

## 2. Adresse IP source

```
🎯 192.168.1.100
```

> ⚠️ Il s'agit d'une adresse **RFC 1918 (réseau privé)**. Cela signifie que l'attaque provient de l'**intérieur du réseau local**, ce qui est particulièrement préo

# 🔍 Analyse du log SSH

## 1. Type d'attaque potentielle

### **Attaque par force brute (Brute Force Attack) sur SSH**

Plusieurs indicateurs convergent :

| Indicateur | Détail | Niveau de suspicion |
|---|---|---|
| **Répétition rapide** | 5 tentatives en **8 secondes** (~1 toutes les 2s) | 🔴 Élevé |
| **Compte ciblé** | `root` — le compte le plus privilégié | 🔴 Critique |
| **Échecs systématiques** | 100% de `Failed password` | 🔴 Élevé |
| **Ports source incrémentaux** | 54322 → 54324 → 54326 → 54328 → 54330 (+2) | 🔴 Automatisation |
| **Horaire suspect** | **03h23 du matin** — heure creuse typique | 🟡 Moyen |

> L'incrémentation régulière des ports source (+2) est la signature caractéristique d'un **outil automatisé** (type Hydra, Medusa, Ncrack) et non d'un humain.

---

## 2. Adresse IP source

```
🎯 192.168.1.100
```

> ⚠️ Il s'agit d'une adresse **RFC 1918 (réseau privé)**. Cela signifie que l'attaque provient de l'**intérieur du réseau local**, ce qui est particulièrement préoccupant :
> - Machine interne compromise (pivot d'attaque)
> - Employé malveillant (menace interne)
> - Appareil IoT/BYOD infecté

---

## 3. Actions recommandées

### 🚨 Actions immédiates (< 15 min)

```bash
# Bloquer immédiatement l'IP via iptables
sudo iptables -A INPUT -s 192.168.1.100 -p tcp --dport 22 -j DROP

# Vérifier si une session a finalement réussi
sudo grep "Accepted" /var/log/auth.log | grep "192.168.1.100"

# Vérifier les sessions actives suspectes
sudo ss -tnp | grep "192.168.1.100"
who -u
```

### 🔒 Renforcement SSH (court terme)

```bash
# /etc/ssh/sshd_config
PermitRootLogin no              # Désactiver le login root
PasswordAuthentication no        # Imposer l'authentification par clé
MaxAuthTries 3                   # Limiter les tentatives
AllowUsers utilisateur_autorisé  # Liste blanche d'utilisateurs
Port 2222                        # Changer le port par défaut (security by obscurity)
LoginGraceTime 30                # Réduire le délai de connexion
```

### 🛡️ Mise en place de protections automatisées

```bash
# Installer et configurer Fail2Ban
sudo apt install fail2ban

# /etc/fail2ban/jail.local
[sshd]
enabled  = true
port     = ssh
maxretry = 3
bantime  = 3600        # Ban d'1 heure
findtime = 300         # Fenêtre de détection de 5 min
```

### 🔎 Investigation approfondie (réseau interne)

```
1. 🖥️  Identifier la machine 192.168.1.100 (DHCP leases, ARP table, inventaire)
2. 🦠  Scanner cette machine pour détecter un malware/compromission
3. 📊  Analyser les logs réseau (firewall, IDS/IPS) pour d'autres activités suspectes
4. 🔗  Vérifier si d'autres machines du rés


# 🔍 Analyse du Log SSH

## 1. Type d'attaque potentielle

### **Attaque par force brute (Brute Force Attack) sur SSH**

Plusieurs indicateurs convergent :

| Indicateur | Détail | Niveau de suspicion |
|---|---|---|
| **Répétition rapide** | 5 tentatives en **8 secondes** (~1 tentative/2s) | 🔴 Critique |
| **Compte ciblé** | `root` — le compte le plus privilégié | 🔴 Critique |
| **Échecs consécutifs** | 100% de `Failed password` | 🔴 Critique |
| **Ports source incrémentaux** | 54322 → 54324 → 54326 → 54328 → 54330 (+2) | 🟠 Élevé |
| **Horaire nocturne** | 03h23 du matin — hors heures ouvrées | 🟠 Élevé |

> L'incrémentation régulière des ports source (+2) suggère l'utilisation d'un **outil automatisé** (type Hydra, Medusa, ou script custom) plutôt qu'une tentative manuelle.

---

## 2. Adresse IP source

```
🎯 192.168.1.100
```

> ⚠️ Il s'agit d'une adresse **RFC 1918 (réseau privé)**. Cela signifie :
> - L'attaque provient du **réseau interne** (LAN)
> - Possibilités : **machine compromise**, **menace interne (insider threat)**, ou **mouvement latéral** d'un attaquant déjà présent dans le réseau

---

## 3. Actions recommandées

### 🚨 Actions immédiates (Court terme)

```bash
# 1. Bloquer immédiatement l'IP via iptables
sudo iptables -A INPUT -s 192.168.1.100 -p tcp --dport 22 -j DROP

# 2. Vérifier si une connexion est toujours active
sudo ss -tnp | grep 192.168.1.100
sudo last | grep 192.168.1.100

# 3. Vérifier l'étendue de l'attaque (autres cibles ?)
sudo grep "192.168.1.100" /var/log/auth.log | tail -50
```

### 🔒 Renforcement SSH (Moyen terme)

```bash
# /etc/ssh/sshd_config
PermitRootLogin no              # Désactiver le login root
PasswordAuthentication no        # Imposer l'authentification par clé
MaxAuthTries 3                   # Limiter les tentatives
AllowUsers utilisateur_autorisé  # Liste blanche d'utilisateurs
Port 2222                        # Changer le port par défaut (optionnel)
```

### 🛡️ Mise en place de protections automatisées

```bash
# Installer et configurer Fail2Ban
sudo apt install fail2ban

# /etc/fail2ban/jail.local
[sshd]
enabled  = true
port     = ssh
maxretry = 3
bantime  = 3600
findtime = 600
```

### 🔎 Investigation approfondie (Prioritaire vu l'IP interne)

```
1. 🖥️  Identifier la machine 192.168.1.100 (DHCP, inventaire réseau)
2. 🔬  Analyser cette machine → est-elle compromise ?
3. 📊  Examiner les logs réseau (firewall, IDS/IPS) pour détecter
       d'autres activités suspectes depuis cette IP
4. 👤  Identifier l'utilisateur/propriétaire de la machine
5. 📝  Documenter l'incident pour le rapport de séc


## 4. Exemple 2 – Analyse de logs firewall

In [38]:
logs_firewall = [
    "2024-01-15 08:15:23 firewall: BLOCKED TCP 10.0.0.5:54321 -> 203.0.113.10:443 (Port Scan)",
    "2024-01-15 08:15:24 firewall: BLOCKED TCP 10.0.0.5:54322 -> 203.0.113.10:443 (Port Scan)",
    "2024-01-15 08:15:25 firewall: BLOCKED TCP 10.0.0.5:54323 -> 203.0.113.10:443 (Port Scan)",
    "2024-01-15 09:30:12 firewall: ALLOWED TCP 192.168.1.10:12345 -> 8.8.8.8:53 (DNS Normal)"
]

prompt_securite = f"""
Analyse ces logs firewall et identifie :
- L'IP qui semble malveillante
- Le type d'attaque
- Le niveau de sévérité (Faible / Moyen / Critique)
- Une recommandation

Logs :
{chr(10).join(logs_firewall)}

Réponds en français de façon claire et structurée.
"""

print(appeler_claude(prompt_securite))


# 🔍 Analyse des Logs Firewall

---

## 1. IP Malveillante Identifiée

| Champ | Détail |
|-------|--------|
| **IP source** | `10.0.0.5` |
| **IP cible** | `203.0.113.10` |
| **Port ciblé** | `443` (HTTPS) |

L'IP `10.0.0.5` est à l'origine de l'activité suspecte. Elle émet **3 connexions TCP consécutives en 2 secondes** depuis des ports source incrémentaux (54321 → 54322 → 54323), toutes bloquées par le firewall.

> L'IP `192.168.1.10` effectue une simple requête DNS vers `8.8.8.8` (Google DNS) → **trafic légitime**, aucune anomalie.

---

## 2. Type d'Attaque

### 🔴 **Scan de ports (Port Scanning)**

Les indices convergent :

- **Connexions rapides et séquentielles** : 3 requêtes en ~2 secondes
- **Ports source incrémentaux** : schéma typique d'un outil de scan automatisé (type **Nmap**, **Masscan**)
- **Même cible et même port** : concentration sur le port 443, probablement pour détecter un service HTTPS actif
- **Mention explicite** du firewall : `(Port Scan)`

Il s'agit d'une phas

# 🔍 Analyse des Logs Firewall

---

## 1. IP Malveillante Identifiée

| Champ | Détail |
|-------|--------|
| **IP source** | `10.0.0.5` |
| **IP cible** | `203.0.113.10` |
| **Port ciblé** | `443` (HTTPS) |

L'IP `10.0.0.5` est à l'origine de l'activité suspecte. Elle émet **3 connexions TCP consécutives en 2 secondes** depuis des ports source incrémentaux (54321 → 54322 → 54323), toutes bloquées par le firewall.

> L'IP `192.168.1.10` effectue une simple requête DNS vers `8.8.8.8` (Google DNS) → **trafic légitime**, aucune anomalie.

---

## 2. Type d'Attaque

### 🔴 **Scan de ports (Port Scanning)**

Les indices convergent :

- **Connexions rapides et séquentielles** : 3 requêtes en ~2 secondes
- **Ports source incrémentaux** : schéma typique d'un outil de scan automatisé (type **Nmap**, **Masscan**)
- **Même cible et même port** : concentration sur le port 443, probablement pour détecter un service HTTPS actif
- **Mention explicite** du firewall : `(Port Scan)`

Il s'agit d'une phase de **reconnaissance** (étape 1 de la kill chain), souvent préalable à une attaque plus ciblée (exploitation de vulnérabilité, brute force, etc.).

---

## 3. Niveau de Sévérité

### ⚠️ **Moyen**

| Facteur | Évaluation |
|---------|------------|
| Les connexions ont été **bloquées** | ✅ Firewall efficace |
| Activité de **reconnaissance** uniquement | Pas d'intrusion constatée |
| Potentiel de **précurseur d'attaque** | ⚠️ Risque d'escalade |
| Volume encore **limité** (3 requêtes) | Pourrait s'intensifier |

> **Justification** : Un scan de ports seul ne cause pas de dommage direct, mais il signale une intention malveillante. S'il n'est pas traité, il peut mener à une attaque de sévérité **critique**.

---

## 4. Recommandations

### Actions immédiates
```
1. 🚫 Bloquer l'IP 10.0.0.5 au niveau du firewall (règle permanente)
2. 📋 Vérifier les logs étendus pour d'autres activités de cette IP
3. 🔍 Contrôler que le serveur 203.0.113.10:443 n'a pas été compromis
```

### Actions préventives
```
4. ⏱️ Configurer un rate-limiting (ex: max 5 connexions/seconde par IP)
5. 🛡️ Activer un IDS/IPS (Snort, Suricata) pour détecter les scans
6. 🔔 Mettre en place des alertes automatiques sur les patterns de scan
7. 📊 Surveiller si l'IP 10.0.0.5 appartient au réseau interne
        → Si oui : machine potentiellement compromise à isoler
        → Si non : vérifier la segmentation réseau
```

---

### 📌 Résumé

| Élément | Conclusion |
|---------|-----------|
| **IP malveillante** | `10.0.0.5` |
| **Attaque** | Scan de ports (reconnaissance) |
| **Sévérité** | ⚠️ **Moyen** |
| **


## 5. Exemple 3 – Extraction d'IOC (Indicateurs de Compromission)

In [39]:
log_incident = """
2024-01-15 14:23:45 web-server: GET /wp-admin/admin-ajax.php?action=malicious HTTP/1.1 200 5432
2024-01-15 14:23:46 web-server: User-Agent: SQLMap/1.6 (https://sqlmap.org)
2024-01-15 14:23:47 db-server: SQL injection attempt detected: ' UNION SELECT * FROM users--
2024-01-15 14:23:48 auth-server: Failed login for 'admin' from 185.130.5.253
Source IP: 185.130.5.253
Target: 10.0.0.15
"""

prompt_ioc = f"""
Extrait les Indicateurs de Compromission (IOC) de ce log :
- Adresses IP
- Noms d'utilisateurs ciblés
- Types d'attaques
- Outils utilisés par l'attaquant

Log :
{log_incident}

Réponds en français.
"""

print(appeler_claude(prompt_ioc))


# Extraction des Indicateurs de Compromission (IOC)

---

## 🔴 Adresses IP

| Adresse IP | Rôle |
|---|---|
| **185.130.5.253** | IP source de l'attaquant |
| **10.0.0.15** | IP cible (serveur interne) |

---

## 👤 Noms d'utilisateurs ciblés

| Utilisateur | Contexte |
|---|---|
| **admin** | Tentative de connexion échouée (brute force / credential stuffing) |

---

## ⚔️ Types d'attaques identifiés

| Type d'attaque | Détails |
|---|---|
| **Injection SQL** | Payload détecté : `' UNION SELECT * FROM users--` — tentative d'exfiltration de la table `users` |
| **Brute force / Tentative d'authentification** | Échec de connexion sur le compte `admin` depuis une IP externe |
| **Exploitation WordPress** | Requête suspecte vers `/wp-admin/admin-ajax.php` avec un paramètre `action=malicious` |

---

## 🛠️ Outils utilisés par l'attaquant

| Outil | Version | Description |
|---|---|---|
| **SQLMap** | 1.6 | Outil automatisé d'injection SQL (identifié via le User-Agent : `SQLMap/1.6`) |

---

#

# Extraction des Indicateurs de Compromission (IOC)

---

## 🔴 Adresses IP

| Adresse IP | Rôle |
|---|---|
| **185.130.5.253** | IP source de l'attaquant |
| **10.0.0.15** | IP cible (serveur interne) |

---

## 👤 Noms d'utilisateurs ciblés

| Utilisateur | Contexte |
|---|---|
| **admin** | Tentative de connexion échouée (brute force / credential stuffing) |

---

## ⚔️ Types d'attaques identifiés

| Type d'attaque | Détails |
|---|---|
| **Injection SQL** | Payload détecté : `' UNION SELECT * FROM users--` — tentative d'exfiltration de la table `users` |
| **Brute force / Tentative d'authentification** | Échec de connexion sur le compte `admin` depuis une IP externe |
| **Exploitation WordPress** | Requête suspecte vers `/wp-admin/admin-ajax.php` avec un paramètre `action=malicious` |

---

## 🛠️ Outils utilisés par l'attaquant

| Outil | Version | Description |
|---|---|---|
| **SQLMap** | 1.6 | Outil automatisé d'injection SQL (identifié via le User-Agent : `SQLMap/1.6`) |

---

## 📋 Résumé de l'incident

> L'attaquant depuis **185.130.5.253** a utilisé **SQLMap v1.6** pour mener une **injection SQL** de type `UNION-based` contre un serveur WordPress (**10.0.0.15**), ciblant spécifiquement la **table `users`** de la base de données. En parallèle, une **tentative de connexion** sur le compte **admin** a été enregistrée. L'ensemble de l'attaque s'est déroulé le **15 janvier 2024 à 14h23**.


## 6. Exemple 4 – Rapport quotidien automatisé

In [ ]:
logs_quotidiens = """
[2024-01-15 00:15:32] SSH: Accepted password for john from 192.168.1.100
[2024-01-15 00:15:33] SSH: Accepted password for john from 192.168.1.100
[2024-01-15 02:30:15] FIREWALL: Port scan detected from 45.33.22.11
[2024-01-15 02:30:16] FIREWALL: Blocked 45.33.22.11 - 100 connection attempts in 5 seconds
[2024-01-15 03:45:22] WEB: GET /backup.zip - 404 Not Found
[2024-01-15 03:45:23] WEB: GET /backup.zip - 404 Not Found
[2024-01-15 03:45:24] WEB: GET /backup.zip - 404 Not Found
[2024-01-15 08:20:11] DB: MySQL error - Access denied for user 'root'@'10.0.0.50'
[2024-01-15 08:20:12] DB: MySQL error - Access denied for user 'root'@'10.0.0.50'
[2024-01-15 08:20:13] DB: MySQL error - Access denied for user 'root'@'10.0.0.50'
"""

def analyser_logs_quotidiens(logs: str) -> str:
    """Génère un rapport SOC structuré à partir des logs."""
    prompt = f"""
    Tu es un analyste SOC. Analyse ces logs et fournis :
    1. Un résumé des incidents détectés (maximum 3)
    2. Classification par sévérité (CRITIQUE, ÉLEVÉE, MOYENNE, FAIBLE)
    3. Recommandations immédiates
    4. IPs à blacklister

    Logs :
    {logs}
    """
    return appeler_claude(prompt, max_tokens=1500)

rapport = analyser_logs_quotidiens(logs_quotidiens)
print(rapport)


## 7. Système d'alerte en temps réel

Chaque ligne de log est évaluée individuellement ; la réponse est parsée en JSON robuste.


In [40]:
import json

In [ ]:
def alerte_securite(log_ligne: str) -> dict:
    """
    Évalue une ligne de log et retourne un dict d'alerte.
    Gère proprement les cas où le modèle ajoute du texte autour du JSON.
    """
    prompt = f"""
    Analyse cette ligne de log.
    Réponds UNIQUEMENT par un objet JSON, sans texte supplémentaire, sans backticks :
    {{"alerte": "oui", "niveau": "critique", "raison": "description courte"}}
    ou {{"alerte": "non", "niveau": "", "raison": ""}}

    Log : {log_ligne}
    """
    reponse_brute = appeler_claude(prompt, max_tokens=200)

    # ✅ Extraction robuste du JSON même si le modèle ajoute du texte autour
    match = re.search(r'\{.*?\}', reponse_brute, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    # Fallback en cas d'échec de parsing
    return {"alerte": "inconnu", "niveau": "", "raison": reponse_brute.strip()}


logs_test = [
    "SSH: Failed password for root from 1.2.3.4",
    "FIREWALL: Normal connection to google.com",
    "DB: SQL injection detected in query",
    "WEB: 404 Not Found for /images/logo.png"
]

for log in logs_test:
    resultat = alerte_securite(log)
    emoji = "🔴" if resultat.get("alerte") == "oui" else "🟢"
    print(f"{emoji} Log    : {log}")
    print(f"   Alerte : {resultat}")
    print("-" * 60)


## 8. Sauvegarde des résultats

In [41]:
from datetime import datetime

In [ ]:
def sauvegarder_analyse(logs: str, analyse: str, nom_fichier: str = "analyse_logs.json") -> None:
    """Sauvegarde les logs et l'analyse dans un fichier JSON."""
    # ✅ Troncature propre sur une limite de ligne, pas au milieu d'un caractère
    MAX_CHARS = 500
    if len(logs) > MAX_CHARS:
        derniere_ligne = logs[:MAX_CHARS].rfind("\n")
        logs_tronques = logs[:derniere_ligne] + "\n... (tronqué)"
    else:
        logs_tronques = logs

    resultat = {
        "date": datetime.now().isoformat(),
        "logs_analyses": logs_tronques,
        "analyse": analyse,
        "statut": "complet"
    }

    with open(nom_fichier, "w", encoding="utf-8") as f:
        json.dump(resultat, f, indent=2, ensure_ascii=False)

    print(f"✓ Analyse sauvegardée dans {nom_fichier}")

sauvegarder_analyse(logs_quotidiens, rapport)


## 9. 🎯 Défi final – Créez votre propre analyseur

Complétez la fonction `analyse_personnalisee` ci-dessous pour :
1. Détecter automatiquement les patterns suspects
2. Générer un rapport JSON structuré
3. Proposer des actions correctives


In [ ]:
def analyse_personnalisee(fichier_logs: str) -> str:
    """
    À COMPLÉTER PAR L'ÉLÈVE

    Créez une analyse qui :
    1. Détecte automatiquement les patterns suspects
    2. Génère un rapport JSON structuré
    3. Propose des actions correctives
    """
    prompt = f"""
    Voici des logs à analyser :
    {fichier_logs}

    Générez un rapport de sécurité complet avec :
    - Résumé exécutif
    - Détails des incidents
    - Niveau de risque global (CRITIQUE / ÉLEVÉ / MOYEN / FAIBLE)
    - Recommandations prioritaires

    Format : Réponse en français, structurée et professionnelle.
    """
    return appeler_claude(prompt, max_tokens=2000)

# ─── Testez votre analyseur ───────────────────────────────────────────────────
logs_defi = """
2024-01-15 22:15:33 - ALERTE: Tentative d'intrusion détectée sur le port 3389
2024-01-15 22:15:34 - SOURCE IP: 5.5.5.5
2024-01-15 22:15:35 - Multiple connexions RDP échouées
"""

resultat_defi = analyse_personnalisee(logs_defi)
print(resultat_defi)


---
## ✨ Fin du TP – Félicitations !

### Compétences acquises
| # | Compétence |
|---|-----------|
| ✅ | Appel d'API LLM avec Python (SDK Anthropic) |
| ✅ | Gestion sécurisée de la clé API (variable d'environnement) |
| ✅ | Analyse de logs SSH, firewall, base de données |
| ✅ | Extraction d'indicateurs de compromission (IOC) |
| ✅ | Parsing JSON robuste des réponses du modèle |
| ✅ | Création de systèmes d'alerte en temps réel |
| ✅ | Génération et sauvegarde de rapports automatisés |
